# 9.4. Recurrent Neural Networks
D2L의 Recurrent Neural Networks장을 PyTorch 기준으로 정리함.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 왜 RNN이 필요한가?

앞에서 n-gram 언어 모델을 배웠다. 예를 들어서 trigram은 다음 단어를 예측할 때 이전 두 단어만 사용한다.

$$
P(x_t \mid x_{t-1}, x_{t-2})
$$

더 긴 문맥을 보고 싶다면 $n$을 크게 만들면 된다. 하지만 vocabulary 크기가 $|\mathcal{V}|$ 라면 n-gram에서 고려해야 하는 조합의 수는 대략 이렇게 매우 빠르게 증가한다.

$$
|\mathcal{V}|^n
$$

예를 들어 vocabulary가 10,000개라면 이렇다.

```text
Bigram  : 10,000²
Trigram : 10,000³
4-gram  : 10,000⁴
```

과거의 모든 토큰을 직접 저장하는 대신, 과거 정보를 하나의 벡터로 요약해서 저장하는 방법을 생각할 수 있다. 그것이 RNN의 핵심인 hidden state다.

## 2. Hidden State

RNN에서는 과거의 모든 토큰을 직접 사용하는 대신 이렇게 표현한다

$$
P(x_t \mid x_{t-1}, \dots, x_1)
\approx
P(x_t \mid h_{t-1})
$$

여기서 $h_{t-1}$ 은 지금까지 들어온 sequence의 정보를 저장하고 있는 hidden state다.

```text
x₁, x₂, x₃, ... xₜ₋₁
          ↓
       hₜ₋₁
          ↓
    다음 토큰 xₜ 예측
```

hidden state는 현재 입력과 이전 hidden state를 이용해서 다시 계산된다.

$$
h_t = f(x_t, h_{t-1})
$$

현재 상태는 `현재 입력 + 이전까지 기억하고 있던 정보`를 조합해서 만들어진다.

## 3. Hidden layer와 Hidden State는 다르다.

이름이 비슷해 혼동하기 쉽다.

### Hidden Layer

MLP에서 배웠던 은닉층이다.

    input -> Hidden Layer -> Output

입력층과 출력층 사이에 존재하는 layer라는 의미이다.

### Hidden State

RNN에서 이전 time step의 정보를 가지고 있는 상태값이다.

    hₜ₋₁ → hₜ → hₜ₊₁

RNN에서는 hidden layer의 출력이 다음 time step으로 전달되면서 hidden state 역할까지 한다.

## 4. 일반적인 MLP부터 봐보자.

일반적인 MLP의 hidden layer를 생각해보자.

입력 minibatch가 이렇다고 해보자

$$
\mathbf{X} \in \mathbb{R}^{n \times d}
$$

$n$: batch size  
$d$: 입력 feature 수

hidden unit의 개수가 $h$라면

$$
\mathbf{W}_{xh}
\in
\mathbb{R}^{d\times h}
$$

이고 hidden layer 출력은 이렇다.
$$
\phi(
\mathbf{X}\mathbf{W}_{xh}
+
\mathbf{b}_h
)
$$


이전에 배웠던 $WX+b$ 구조와 같다. 여기서 $\phi$는 activation function이다. 예를 들어서 tanh나 ReLU가 사용될 수 있다.

## 5. MLP의 출력 계산

hidden layer에서

$$
\mathbf H
$$

를 얻었다면 출력은

$$
\mathbf H\mathbf W_{hq}
+
\mathbf b_q
$$

로 계산한다.

구조를 단순하게 표현하면 이렇다.

```text
X
↓
XW_xh + b_h
↓
Activation
↓
H
↓
HW_hq + b_q
↓
O
```

분류 문제라면 마지막 $\mathbf O$에 softmax를 적용해서 각 class의 확률을 구할 수 있다. 문제는 이 구조가 각각의 입력을 독립적으로 처리한다는 것이다. 이전 입력에 대한 기억이 없다.

## 6. RNN에서는 무엇이 추가될까?

RNN의 핵심 공식은 다음과 같다.
$$
\phi(
\mathbf X_t\mathbf W_{xh}
+
\mathbf H_{t-1}\mathbf W_{hh}
+
\mathbf b_h
)
$$

MLP와 비교해보자.

MLP
$$
\phi(
\mathbf X\mathbf W_{xh}
+
\mathbf b_h
)
$$

RNN
$$
\phi(
\mathbf X_t\mathbf W_{xh}
+
\mathbf H_{t-1}\mathbf W_{hh}
+
\mathbf b_h
)
$$

딱 하나가 추가됐다.

$$
\boxed{
\mathbf H_{t-1}\mathbf W_{hh}
}
$$

이 항이 이전 time step의 정보를 현재 계산에 넣어주는 부분이다. RNN의 핵심은 정말 단순하게 보면 이렇다.

    현재 입력 + 이전 기억 -> 현재 기억

## 7. RNN의 Weight 세 가지

RNN에는 중요한 weight 세 종류가 있다.

### 1. 입력 -> Hidden

$$
\mathbf W_{xh}
$$

현재 입력 $\mathbf X_t$를 hidden state로 변환한다.

    Xₜ → Hₜ
### 2. Hidden → Hidden

$$
\mathbf W_{hh}
$$

이전 hidden state를 현재 hidden state 계산에 사용한다.

     Hₜ₋₁ → Hₜ

이 weight가 RNN을 RNN답게 만드는 가장 중요한 parameter다.

### 3. Hidden → Output

$$
\mathbf W_{hq}
$$

hidden state를 실제 출력으로 변환한다.

    Hₜ → Oₜ

전체적으로 보면 이렇게 생겼다.

```text
Xₜ ──W_xh──┐
            ↓
           Hₜ ──W_hq──→ Oₜ
            ↑
Hₜ₋₁─W_hh──┘
```

## 8. RNN이 Sequence를 기억하는 과정

예를 들어 다음 문자가 들어온다고 해보자.

    m -> a -> c -> h -> i -> n -> e

처음 m을 입력한다.

$$
\phi(
X_1W_{xh}
+
H_0W_{hh}
+
b_h
)
$$

처음에는 일반적으로 $H_0$를 0으로 초기화할 수 있다.

다음 a가 들어오면
$$
\phi(
X_2W_{xh}
+
H_1W_{hh}
+
b_h
)
$$

여기서 $H_1$에는 이미 m에 대한 정보가 포함되어 있다.

다음 c에서는 이렇다.
$$
\phi(
X_3W_{xh}
+
H_2W_{hh}
+
b_h
)
$$

그런데 $H_2$에는 이전에 들어온

m → a

의 정보가 들어있다.

따라서 $H_3$에는 간접적으로

m → a → c

의 정보가 반영될 수 있다.

결국 이렇게 sequence의 과거 정보가 전달된다.

```text
H₁ : m의 정보
H₂ : m, a의 정보
H₃ : m, a, c의 정보
H₄ : m, a, c, h의 정보
```

정확히 말하면 모든 토큰을 그대로 저장하는 것이 아니라, 학습에 필요한 형태의 벡터 표현으로 압축해서 전달한다.

## 9. RNN의 출력

현재 hidden state를 얻었다면 출력은 MLP와 거의 같다.
$$
\mathbf H_t\mathbf W_{hq}
+
\mathbf b_q
$$

그래서 한 time step에서 전체 계산은 이렇다.
$$
\phi(
\mathbf X_t\mathbf W_{xh}
+
\mathbf H_{t-1}\mathbf W_{hh}
+
\mathbf b_h
)
$$
$$
\mathbf H_t\mathbf W_{hq}
+
\mathbf b_q
$$

구조는 다음과 같다.

```text
           Hₜ₋₁
             │
             ↓
Xₜ ───────→ Hₜ ───────→ Oₜ
             │
             ↓
           Hₜ₊₁
```

$H_t$는 두 가지 역할을 한다.

1. Oₜ를 계산하는 데 사용
2. 다음 Hₜ₊₁을 계산하는 데 사용

## 10. 모든 Time Step에서 같은 Weight를 사용한다.

RNN에서 매우 중요한 특징이다.

예를 들어서 `m -> a -> c -> h -> i -> n -> e` 을 처리하더라도 매 문자마다 새로운 weight를 만드는 게 아니다.

모든 time step에서 
$$
W_{xh},\quad W_{hh},\quad W_{hq}
$$

이것을 동일하게 사용한다.

```text
t=1 : W_xh, W_hh, W_hq
t=2 : W_xh, W_hh, W_hq
t=3 : W_xh, W_hh, W_hq
...
```

이렇게 parameter sharing이 이루어진다. 그래서 sequence가 길어진다고 해서 RNN의 parameter 개수가 증가하지 않는다.

예를 들어 길이가 이렇게 증가해도 사용하는 weight 자체는 동일하다.

    10 -> 100 -> 1000

단지 같은 RNN 연산을 더 많이 반복할 뿐이다. 이것이 n-gram과 비교했을 때 매우 중요한 장점이다.

## 11 입력은 Hidden State를 Concatenate해서 볼 수도 있다.

RNN 계산은 이렇다.

$$
X_tW_{xh}
+
H_{t-1}W_{hh}
$$

그런데 이것은 사실 입력과 hidden state를 합친 뒤 한 번의 행렬곱을 하는 것과 동일하게 표현할 수 있다.

예를 들어서

```python
import torch

X = torch.randn(3, 1)
W_xh = torch.randn(1, 4)

H = torch.randn(3, 4)
W_hh = torch.randn(4, 4)
```

각각 계산하면 이렇다.

    torch.matmul(X, W_xh) + torch.matmul(H, W_hh)

shape을 보면

```text
X     : (3, 1)
W_xh  : (1, 4)

H     : (3, 4)
W_hh  : (4, 4)

X @ W_xh → (3, 4)

H @ W_hh → (3, 4)
```

둘을 더해서 (3, 4) 를 얻는다. 그런데 다음처럼 concatenate할 수도 있다.

```python
torch.matmul(
    torch.cat((X, H), dim=1),
    torch.cat((W_xh, W_hh), dim=0)
)
```
shape은

[X, H]
(3, 1) + (3, 4) -> (3, 5)

[W_xh, W_hh]
(1, 4) + (4, 4) -> (5, 4)

이므로

$$
(3,5)(5,4)
\rightarrow
(3,4)
$$

결국 두 방식은 같은 계산이다. 개념적으로 RNN을 이렇게 생각해도 된다.

```text
현재 입력 Xₜ
+
이전 hidden state Hₜ₋₁

        ↓ concatenate

[Xₜ, Hₜ₋₁]

        ↓ Fully Connected

Hₜ
```

## 12. RNN으로 문자 수준 언어 모델 만들기

이제 앞에서 배운 언어 모델과 RNN을 결합할 수 있다.

예를 들어 `machine` 이라는 문자열이 있다고 하자. 입력 sequence는

    m a c h i n

정답 sequence는 한 칸 이동시킨 이거다

    a c h i n e

입력 X : m a c h i n  
정답 Y : a c h i n e

각 time step에서 보면 이걸 학습하는 것이다.

```text
m     → a
ma    → c
mac   → h
mach  → i
machi → n
machin→ e
```

예를 들어 세 번째 time step에서는 현재까지

    m → a → c

를 읽었다.

hidden state에는 이 sequence의 정보가 들어있고 모델은 다음 문자 $h$ 를 예측해야 한다.

## 13. Softmax와 Cross Entropy

각 time step에서 RNN은 $O_t$ 를 출력한다. 문자 수준 언어 모델이라면 출력 차원은 vocabulary 크기와 같다.

예를 들어 사용 가능한 문자가 28개라면 이렇다.

$$
O_t \in \mathbb R^{28}
$$

Softmax를 적용하면

```text
a : 0.02
b : 0.01
c : 0.03
...
h : 0.75
...
```

와 같이 다음 문자에 대한 확률 분포를 얻는다. 정답 문자와 비교하여 Cross Entropy Loss를 계산한다.

```text
RNN Output
    ↓
Softmax
    ↓
다음 문자 확률 분포
    ↓
정답 문자와 비교
    ↓
Cross Entropy Loss
```

그리고 역전파를 통해 아래 같은 parameter가 수정된다.

$$
W_{xh}, W_{hh}, W_{hq}
$$

## 14. RNN 전체 흐름

RNN 언어 모델의 전체 흐름을 정리하면 다음과 같다.

```text
현재 문자 Xₜ
      │
      ↓
   W_xh
      │
      ↓
┌──────────────┐
│              │
│     Hₜ       │──── W_hq ───→ Oₜ
│              │                  │
└──────────────┘                  ↓
      ↑                        Softmax
      │                           │
     W_hh                         ↓
      │                       다음 문자
      │                         예측
    Hₜ₋₁
```

RNN을 이해할 때 일단 이 두 식을 확실히 이해하면 된다.
$$
\phi(
X_tW_{xh}
+
H_{t-1}W_{hh}
+
b_h
)
$$

$$
H_tW_{hq}
+
b_q
$$

## 15. 오늘의 정리

- n-gram은 긴 문맥을 사용하려 할수록 필요한 조합이 폭발적으로 증가한다.
- RNN은 과거 정보를 직접 모두 저장하지 않고 hidden state에 요약해서 전달한다.
- hidden state는 현재 입력과 이전 hidden state를 사용해 계산한다.
- $W_{xh}$는 현재 입력을 hidden state로 전달한다.
- $W_{hh}$는 이전 hidden state를 현재 hidden state로 전달한다.
- $W_{hq}$는 hidden state를 출력으로 전달한다.
- $H_t$에는 현재까지의 sequence 정보가 축적된다.
- $H_t$는 현재 출력 $O_t$를 만드는 동시에 다음 hidden state $H_{t+1}$로 전달된다.
- 모든 time step에서 같은 weight를 재사용한다.
- 따라서 sequence가 길어져도 model parameter 수 자체는 증가하지 않는다.
- 문자 수준 언어 모델에서는 machin → achine처럼 입력과 정답을 한 칸 이동시켜 학습한다.
- 각 time step의 출력에 softmax를 적용하고 다음 문자와 Cross Entropy Loss를 계산한다.